# Адаптерный триминг и фильтрация — человек (`PRJEB30386`)

`cutadapt` удаляет подтверждённые источником Illumina адаптеры read-through, после чего `fastp` фильтрует целые пары (`-q 30 -u 40 -l 250`). Обрезка концов по качеству и автодетекция адаптеров отключены. Каноническая стадия `trimmed` заменяется только после полной проверки промежуточного каталога.

In [ ]:
import os, sys, sysconfig, shutil, subprocess, time, json
from pathlib import Path

_ENV_CANDIDATES = ["/opt/conda/envs/bcr_env", "/Users/epishkin/mamba/envs/bcr_env"]
BCR_ENV = next((Path(p) for p in _ENV_CANDIDATES if (Path(p) / "bin").is_dir()), None)
if BCR_ENV is None:
    raise FileNotFoundError(f"Среда bcr_env не найдена: {_ENV_CANDIDATES}")
os.environ["PATH"] = str(BCR_ENV / "bin") + os.pathsep + os.environ.get("PATH", "")
os.environ["PYTHONNOUSERSITE"] = "1"
for _site in sorted((BCR_ENV / "lib").glob("python*/site-packages")):
    if str(_site) not in sys.path:
        sys.path.insert(0, str(_site))
print("BCR_ENV:", BCR_ENV)
DATASET = "PRJEB30386"
LOCAL_REPO = Path("/Users/epishkin/workspace/bcr-assembler")
VOLUME_ROOT = Path("/data/user/epishkin")

def _has_fastq(path, pattern="*.fastq.gz"):
    return Path(path).is_dir() and any(Path(path).glob(pattern))

def resolve_raw_and_result():
    remote_raw = VOLUME_ROOT / "raw" / DATASET
    local_result = LOCAL_REPO / "results" / DATASET
    local_raw = LOCAL_REPO / "raw" / DATASET
    if _has_fastq(remote_raw, "*_1.fastq.gz"):
        return remote_raw, VOLUME_ROOT / "results" / DATASET
    if _has_fastq(local_raw, "*_1.fastq.gz"):
        return local_raw, local_result
    raise FileNotFoundError(f"Парные FASTQ не найдены в {remote_raw} или {local_raw}")

def resolve_result():
    remote = VOLUME_ROOT / "results" / DATASET
    local = LOCAL_REPO / "results" / DATASET
    return remote if remote.is_dir() else local


In [ ]:
ADAPTER_R1 = "AGATCGGAAGAGCACACGTCTGAACTCCAGTCAC"
ADAPTER_R2 = "AGATCGGAAGAGCGTCGTGTAGGGAAAGAGTGTAG"
ADAPTER_TIMES = 2
ADAPTER_MIN_OVERLAP = 10
QUALITY_PHRED = 30
UNQUALIFIED_PERCENT_LIMIT = 40
MIN_LENGTH = 250
NPROC = 4
FORCE = False

print("cutadapt:", shutil.which("cutadapt"))
print("fastp:", shutil.which("fastp"))


In [ ]:
def _tool(name):
    path = shutil.which(name)
    if not path:
        raise FileNotFoundError(f"Не найден executable: {name} (BCR_ENV={BCR_ENV})")
    return path

def _run_visible(cmd, stdout_log, stderr_log, outputs=(), heartbeat=30):
    stdout_log, stderr_log = Path(stdout_log), Path(stderr_log)
    stdout_log.parent.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    print("[run]", " ".join(map(str, cmd)), flush=True)
    with stdout_log.open("w") as stdout, stderr_log.open("w") as stderr:
        proc = subprocess.Popen([str(x) for x in cmd], stdout=stdout, stderr=stderr, text=True)
        print(f"PID={proc.pid}", flush=True)
        while proc.poll() is None:
            sizes = " ".join(
                f"{Path(x).name}={Path(x).stat().st_size / 1e6:.1f}MB"
                for x in outputs if Path(x).exists()
            )
            print(f"PID={proc.pid} elapsed={(time.monotonic()-started)/60:.1f}min {sizes}", flush=True)
            time.sleep(heartbeat)
    if proc.returncode:
        raise RuntimeError(f"rc={proc.returncode}; см. {stderr_log}")

def _promote(staging, final):
    staging, final = Path(staging), Path(final)
    previous = final.parent / f".{final.name}.previous"
    if previous.exists():
        shutil.rmtree(previous)
    if final.exists():
        final.rename(previous)
    try:
        staging.rename(final)
    except Exception:
        if previous.exists() and not final.exists():
            previous.rename(final)
        raise
    if previous.exists():
        shutil.rmtree(previous)

def run_adapter_trim(force=FORCE):
    raw_dir, result = resolve_raw_and_result()
    final = result / "trimmed"
    staging = result / ".trimmed.staging"
    samples = sorted(p.name.removesuffix("_1.fastq.gz") for p in raw_dir.glob("*_1.fastq.gz"))
    if not samples or any(not (raw_dir / f"{s}_2.fastq.gz").is_file() for s in samples):
        raise RuntimeError(f"Неполный набор парных FASTQ: {raw_dir}")
    if staging.exists():
        if not force:
            raise FileExistsError(f"Остался staging: {staging}; установите FORCE=True для очистки")
        shutil.rmtree(staging)
    out, logs = staging / "fastq", staging / "logs"
    reports, tmp = staging / "fastp_reports", staging / "adapter_only_tmp"
    for d in (out, logs, reports, tmp):
        d.mkdir(parents=True, exist_ok=True)
    summary = {}
    for sample in samples:
        r1, r2 = raw_dir / f"{sample}_1.fastq.gz", raw_dir / f"{sample}_2.fastq.gz"
        a1, a2 = tmp / f"{sample}_1.adapter.fastq.gz", tmp / f"{sample}_2.adapter.fastq.gz"
        o1, o2 = out / f"{sample}_1.trim.fastq.gz", out / f"{sample}_2.trim.fastq.gz"
        cutadapt = [_tool("cutadapt"), "--times", str(ADAPTER_TIMES), "-O", str(ADAPTER_MIN_OVERLAP),
                    "--compression-level", "1", "-a", ADAPTER_R1, "-A", ADAPTER_R2,
                    "--json", logs / f"{sample}.cutadapt.json", "-o", a1, "-p", a2, r1, r2]
        _run_visible(cutadapt, logs / f"{sample}.cutadapt.stdout.log",
                     logs / f"{sample}.cutadapt.stderr.log", (a1, a2))
        report_json, report_html = reports / f"{sample}.fastp.json", reports / f"{sample}.fastp.html"
        fastp = [_tool("fastp"), "-i", a1, "-I", a2, "-o", o1, "-O", o2,
                 "-q", str(QUALITY_PHRED), "-u", str(UNQUALIFIED_PERCENT_LIMIT), "-l", str(MIN_LENGTH),
                 "--disable_adapter_trimming", "--disable_trim_poly_g", "-w", str(NPROC),
                 "-j", report_json, "-h", report_html]
        _run_visible(fastp, logs / f"{sample}.fastp.stdout.log",
                     logs / f"{sample}.fastp.stderr.log", (o1, o2))
        data = json.loads(report_json.read_text())
        before = data["summary"]["before_filtering"]["total_reads"]
        after = data["summary"]["after_filtering"]["total_reads"]
        summary[sample] = {"before_pairs": before // 2, "after_pairs": after // 2,
                           "retention": round(after / before, 6) if before else 0.0}
        a1.unlink(); a2.unlink()
    expected = {f"{s}_{mate}.trim.fastq.gz" for s in samples for mate in (1, 2)}
    if {p.name for p in out.glob("*.trim.fastq.gz")} != expected:
        raise RuntimeError("Проверка комплекта trimmed FASTQ не пройдена")
    if len(list(reports.glob("*.fastp.json"))) != len(samples):
        raise RuntimeError("Не хватает fastp JSON")
    shutil.rmtree(tmp)
    (staging / "filter_summary.json").write_text(json.dumps({
        "adapters": {"R1": ADAPTER_R1, "R2": ADAPTER_R2},
        "cutadapt": {"times": ADAPTER_TIMES, "minimum_overlap": ADAPTER_MIN_OVERLAP},
        "fastp": {"qualified_quality_phred": QUALITY_PHRED,
                  "unqualified_percent_limit": UNQUALIFIED_PERCENT_LIMIT,
                  "minimum_length": MIN_LENGTH, "quality_end_trimming": False},
        "samples": summary}, ensure_ascii=False, indent=2) + "\n")
    _promote(staging, final)
    print("Готово:", final)


## Запуск

По умолчанию `FORCE=False`: существующая незавершённая промежуточная стадия не удаляется автоматически. Для осознанного перезапуска установите `FORCE=True`.

In [ ]:
run_adapter_trim(force=FORCE)
